# EExPolytech-2026

Три прогона, обсуждаемые в разделе Results:

| Рисунок | Прогон | Каталог эксперимента | Особенность |
|---|---|---|---|
| Fig. 2 | run 1 | `2026-03-12_15-39-27_neural_controller-v3` | малый $\Delta u_{max} = 20$, $\gamma = 0.99$; режима оценивания нет |
| Fig. 3 | run 2 | `2026-03-26_16-54-41_neural_controller-v3` | увеличенный $\Delta u_{max} = 100$, обычная целевая энтропия ($-\dim a = -1$) |
| Fig. 4 | run 3 | `2026-06-05_14-36-04_neural_controller-v3` | та же конфигурация, что и run 2, но целевая энтропия понижена до $-5$; ось времени обрезана до 180 мин |


## Настройка: палитра, шрифты, каталог вывода

In [ ]:
import matplotlib.pyplot as plt

from nn_laser_stabilizer.analysis.experiments import (
    ExperimentId,
    NeuralControllerExperiment,
)

# Палитра Okabe-Ito
PHOTODETECTOR = "#0072B2"
BAND_ALPHA = 0.18
SETPOINT = "#D55E00"
CONTROL = "#009E73"
HANDOVER = "#000000"
EVAL_FILL = "#999999"
EVAL_ALPHA = 0.15

In [ ]:
COL_W = 3.39 # ширина колонки IEEE A4, дюймы (86 мм)

plt.rcParams.update({
    "font.size": 8,
    "axes.labelsize": 8,
    "xtick.labelsize": 8,
    "ytick.labelsize": 8,
    "legend.fontsize": 7,
    "axes.linewidth": 0.6,
    "xtick.major.width": 0.6,
    "ytick.major.width": 0.6,
    "savefig.dpi": 300,
    "font.family": "serif",
    "font.serif": ["Times New Roman"]
})

In [ ]:
from functools import wraps
from pathlib import Path

OUTPUT_DIR = Path("output") / "eexpolytech-2026"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

if not getattr(plt.savefig, "_auto_wrapped", False):
    _original_savefig = plt.savefig

    @wraps(_original_savefig)
    def _auto_savefig(filename, *args, **kwargs):
        filename = OUTPUT_DIR / Path(filename).name
        return _original_savefig(filename, *args, **kwargs)

    setattr(_auto_savefig, "_auto_wrapped", True)

    plt.savefig = _auto_savefig

## Функция построения рисунка

Верхняя панель (a) — скользящее среднее сигнала фотодетектора с полосой ±СКВО,
нижняя (b) — сырой управляющий сигнал. Легенда собирается из фактически
нарисованных элементов, поэтому у прогона без режима оценивания
соответствующая запись отсутствует сама собой.

In [ ]:
def plot_run(dir_name: str, out_stem: str, xlim: tuple | None = None, window: int = 5000):
    """Построить пару панелей (фотодетектор / управление) для одного прогона.

    dir_name — каталог эксперимента, out_stem — имя файлов в output/ без расширения,
    xlim — ограничение оси времени в минутах, window — окно скользящего среднего в шагах.

    Фигура строится сразу в конечном размере: ширина COL_W соответствует колонке
    шаблона IEEE A4 (86 мм), поэтому в Word рисунок вставляется без масштабирования.
    У прогонов инференса фазы ПИД нет, и линия передачи управления не рисуется.
    """
    exp = NeuralControllerExperiment(ExperimentId.from_dir_name(dir_name))
    ia = exp.interaction

    setpoint = float(exp.params.setpoint)

    # Ось времени: global_step пересчитывается в минуты по полной длительности прогона.
    t_min = ia.time.minutes
    experiment_time_min = float(t_min.iloc[-1])
    x_max = float(ia.global_step.max())

    def global_step_to_min(gs: float) -> float:
        return gs / x_max * experiment_time_min

    pid_end = ia.phases.pid.end
    handover_min = global_step_to_min(pid_end.steps) if pid_end is not None else None
    eval_ranges_min = [
        (global_step_to_min(x0), global_step_to_min(x1))
        for x0, x1 in ia.phases.eval.ranges
    ]

    rolling_mean = ia.env.process_variable.rolling(window).mean()
    rolling_std = ia.env.process_variable.rolling(window).std()

    fig, (ax_pv, ax_co) = plt.subplots(
        2,
        1,
        figsize=(COL_W, 2.6),
        sharex=True,
        constrained_layout=True,
        gridspec_kw={"height_ratios": [1, 1]},
    )
    fig.get_layout_engine().set(hspace=0.04, h_pad=0.02, w_pad=0.02)

    # (a) сигнал фотодетектора
    ax_pv.fill_between(
        t_min,
        rolling_mean - rolling_std,
        rolling_mean + rolling_std,
        color=PHOTODETECTOR,
        alpha=BAND_ALPHA,
        linewidth=0,
        label="±std",
        rasterized=True,
    )

    ax_pv.plot(t_min, rolling_mean, color=PHOTODETECTOR, linewidth=1.0)

    ax_pv.axhline(
        y=setpoint,
        linestyle="--",
        linewidth=1.2,
        color=SETPOINT,
        label="Setpoint",
    )

    if handover_min is not None:
        ax_pv.axvline(
            x=handover_min,
            color=HANDOVER,
            linestyle="--",
            linewidth=1.2,
            label="Transition to NN control",
        )

    for i, (x0, x1) in enumerate(eval_ranges_min):
        ax_pv.axvspan(
            x0,
            x1,
            color=EVAL_FILL,
            alpha=EVAL_ALPHA,
            linewidth=0,
            label="Evaluation Mode" if i == 0 else None,
        )

    # (b) управляющий сигнал
    ax_co.plot(
        t_min,
        ia.env.control_output,
        color=CONTROL,
        linewidth=0.5,
        rasterized=True,
    )

    if handover_min is not None:
        ax_co.axvline(x=handover_min, color=HANDOVER, linestyle="--", linewidth=1.2)

    for x0, x1 in eval_ranges_min:
        ax_co.axvspan(x0, x1, color=EVAL_FILL, alpha=EVAL_ALPHA, linewidth=0)

    ax_pv.set_ylabel("Photodetector signal\n(ADC counts)")
    ax_co.set_ylabel("Control signal\n(DAC counts)")
    ax_co.set_xlabel("Time (min)")

    fig.align_ylabels([ax_pv, ax_co])

    ax_pv.grid(False)
    ax_co.grid(False)

    ax_pv.text(
        -0.18, 1.10, "(a)",
        transform=ax_pv.transAxes,
        fontsize=8,
        va="bottom",
        ha="center",
    )

    ax_co.text(
        -0.18, 1.02, "(b)",
        transform=ax_co.transAxes,
        fontsize=8,
        va="bottom",
        ha="center",
    )

    handles, labels = ax_pv.get_legend_handles_labels()
    ax_pv.legend(
        handles,
        labels,
        loc="lower center",
        bbox_to_anchor=(0.5, 1.02),
        ncol=2,
        frameon=False,
        fontsize=7,
        handlelength=1.6,
        columnspacing=1.0,
        borderpad=0.2,
        labelspacing=0.3,
    )

    if xlim is not None:
        ax_co.set_xlim(*xlim)

    plt.savefig(f"{out_stem}.pdf", bbox_inches="tight", pad_inches=0.01)
    plt.savefig(f"{out_stem}.svg", bbox_inches="tight", pad_inches=0.01)
    plt.savefig(f"{out_stem}.png", bbox_inches="tight", pad_inches=0.01, dpi=300)

    plt.show()
    return exp

## Числовые метрики

In [ ]:
import numpy as np
import pandas as pd


def error_metrics(exp: NeuralControllerExperiment, *, tail: int = 100_000) -> pd.DataFrame:
    """MAE и RMSE ошибки слежения для фаз ПИД и НС — числа для тезисов.

    ПИД считается по всей фазе инициализации, НС — по последним ``tail``
    точкам прогона. Единицы — отсчёты АЦП (как на панелях сигнала выше).
    """
    ia = exp.interaction
    pid_error = ia.observations.error[ia.phases.pid.mask]
    nn_error = ia.observations.error[ia.phases.nn.mask].iloc[-tail:]

    def row(name: str, e: pd.Series) -> dict:
        return {
            "Controller": name,
            "N points": len(e),
            "MAE": e.abs().mean(),
            "RMSE": np.sqrt((e ** 2).mean()),
        }

    return pd.DataFrame(
        [row("PID", pid_error), row("NN", nn_error)]
    ).set_index("Controller")

## Fig. 2 — run 1: малый максимальный шаг

Режим оценивания в этом прогоне не запускался.

In [ ]:
exp1 = plot_run(
    "2026-03-12_15-39-27_neural_controller-v3",
    out_stem="eexpolytech_fig2_run1",
)

In [ ]:
error_metrics(exp1)

## Fig. 3 — run 2: увеличенный максимальный шаг

Максимальный шаг поднят до 100, добавлен детерминированный режим оценивания (серые полосы). 

In [ ]:
exp2 = plot_run(
    "2026-03-26_16-54-41_neural_controller-v3",
    out_stem="eexpolytech_fig3_run2",
)

In [ ]:
error_metrics(exp2)

## Fig. 4 — run 3: пониженная целевая энтропия

Та же конфигурация, что и в run 2, но целевая энтропия понижена до −5.

In [ ]:
exp3 = plot_run(
    "2026-06-05_14-36-04_neural_controller-v3",
    out_stem="eexpolytech_fig4_run3",
    xlim=(0, 230),
)

In [ ]:
error_metrics(exp3)

In [ ]:
summary = pd.concat(
    {
        "run 1": error_metrics(exp1),
        "run 2": error_metrics(exp2),
        "run 3": error_metrics(exp3),
    },
    names=["Run", "Controller"],
)
summary

In [ ]:
result = (
    summary.drop(columns="N points")
    .stack()
    .rename_axis(["Run", "Controller", "Metric"])
    .unstack("Controller")
    .reset_index()
)
result[["PID", "NN"]] = result[["PID", "NN"]].round(1)

result

## Замечания рецензентов

### Вопрос 2: прогоны с пониженной целевой энтропией

Успешные прогоны с $H_{target} = -5$: 4 обучающих прогона с одинаковой конфигурацией
и 9 прогонов инференса актора, обученного в run 3. Прогон с $H_{target} = -3$
и неудачные запуски (`2026-06-10_17-50-46` — сбой передачи данных,
`2026-06-19_15-15-53` — PV застрял на одном значении) не рассматриваются.

Цель — по графикам понять, достигнута ли стабилизация. У инференса весь прогон —
детерминированный режим, поэтому он целиком закрашен серым.

In [ ]:
TRAIN_RUNS_H5 = [
    "2026-06-05_14-36-04_neural_controller-v3",  # run 3
    "2026-06-08_17-18-47_neural_controller-v3",
    "2026-06-09_13-36-37_neural_controller-v3",
    "2026-06-09_17-49-47_neural_controller-v3",
]

for dir_name in TRAIN_RUNS_H5:
    plot_run(dir_name, out_stem=f"review_{dir_name}")

In [ ]:
INFERENCE_RUNS = [
    "2026-06-10_19-15-43_neural-controller-v3_inference",
    "2026-06-15_16-49-32_neural-controller-v3_inference",
    "2026-06-16_17-13-25_neural-controller-v3_inference",
    "2026-06-16_19-34-08_neural-controller-v3_inference",
    "2026-06-17_15-47-34_neural-controller-v3_inference",
    "2026-06-17_18-02-15_neural-controller-v3_inference",
    "2026-06-18_14-28-06_neural-controller-v3_inference",
    "2026-06-18_15-36-39_neural-controller-v3_inference",
    "2026-06-19_15-57-05_neural-controller-v3_inference",
]

In [ ]:
for dir_name in INFERENCE_RUNS:
    plot_run(dir_name, out_stem=f"review_{dir_name}")

### Вопрос 1: сбросы

Сброс — выход управляющего сигнала на границу диапазона, по которому завершается эпизод.
Длительность — сумма измеренных интервалов между шагами. Нужны `exp1`–`exp3` из ячеек выше.

In [ ]:
def resets_and_duration(exp: NeuralControllerExperiment, mask: pd.Series) -> dict:
    """Число сбросов (выход управления на границу диапазона) и измеренная длительность участка."""
    ia = exp.interaction
    return {
        "Resets": int(ia.terminated[mask].astype(bool).sum()),
        "Duration (min)": ia.step_interval.seconds[mask].sum() / 60,
    }


def final_eval_mask(exp: NeuralControllerExperiment) -> pd.Series:
    """Маска последнего (финального) участка детерминированного оценивания."""
    ia = exp.interaction
    start, end = ia.phases.eval.ranges[-1]
    return (ia.global_step >= start) & (ia.global_step <= end)


retention = pd.DataFrame(
    {
        "run 1 (NN, learning)": resets_and_duration(exp1, exp1.interaction.phases.nn.mask),
        "run 2 (final evaluation)": resets_and_duration(exp2, final_eval_mask(exp2)),
        "run 3 (final evaluation)": resets_and_duration(exp3, final_eval_mask(exp3)),
    }
).T
retention

### Коллаж инференса для слайда

Все 9 успешных прогонов инференса актора run 3 (список `INFERENCE_RUNS` выше), сетка 3×3
под слайд 16:9. Над каждым графиком — дата, время запуска и уставка: актор обучен при уставке 120,
часть прогонов запущена при 110 и 100. Шкалы по вертикали общие для всех графиков, по времени —
своя у каждого. Начальный участок до выхода на уставку у части прогонов выходит за шкалу.

In [ ]:
from datetime import datetime


def load_inference(dir_name: str, window: int = 5000) -> dict:
    """Ряды для коллажа: время, скользящее среднее и СКО фотодетектора, управление, уставка."""
    exp = NeuralControllerExperiment(ExperimentId.from_dir_name(dir_name))
    ia = exp.interaction
    pv = ia.env.process_variable
    start = datetime.strptime(f"{exp.id.date} {exp.id.time}", "%Y-%m-%d %H-%M-%S")
    return {
        "label": f"{start:%b %d, %H:%M} · setpoint {exp.params.setpoint:g}",
        "t": ia.time.minutes.to_numpy(),
        "pv_mean": pv.rolling(window).mean().to_numpy(),
        "pv_std": pv.rolling(window).std().to_numpy(),
        "u": ia.env.control_output.to_numpy(),
        "setpoint": exp.params.setpoint,
    }


inference_runs = [load_inference(dir_name) for dir_name in INFERENCE_RUNS]

In [ ]:
SLIDE_SIZE = (13.33, 6.6)  # слайд 16:9 (13.33 × 7.5 дюйма) минус место под заголовок
PV_LIM = (80, 140)
U_LIM = (0, 4095)

with plt.rc_context({
    "font.size": 11,
    "axes.labelsize": 11,
    "axes.titlesize": 11,
    "xtick.labelsize": 10,
    "ytick.labelsize": 10,
    "legend.fontsize": 11,
}):
    fig = plt.figure(figsize=SLIDE_SIZE, layout="constrained")
    outer = fig.add_gridspec(3, 3)

    for k, run in enumerate(inference_runs):
        row, col = divmod(k, 3)
        inner = outer[row, col].subgridspec(2, 1)
        ax_pv = fig.add_subplot(inner[0])
        ax_co = fig.add_subplot(inner[1], sharex=ax_pv)
        t = run["t"]

        ax_pv.fill_between(
            t,
            run["pv_mean"] - run["pv_std"],
            run["pv_mean"] + run["pv_std"],
            color=PHOTODETECTOR,
            alpha=BAND_ALPHA,
            linewidth=0,
            label="±std",
            rasterized=True,
        )
        ax_pv.plot(t, run["pv_mean"], color=PHOTODETECTOR, linewidth=1.0, label="Photodetector signal (moving average)")
        ax_pv.axhline(run["setpoint"], color=SETPOINT, linestyle="--", linewidth=1.2, label="Setpoint")
        ax_co.plot(t, run["u"], color=CONTROL, linewidth=0.5, label="Control signal", rasterized=True)

        ax_pv.set_title(run["label"])
        ax_pv.set_xlim(0, t[-1])
        ax_pv.set_ylim(*PV_LIM)
        ax_co.set_ylim(*U_LIM)
        ax_co.set_yticks([0, 2000, 4000])
        ax_pv.tick_params(labelbottom=False)

        # Подписи осей — только единицы: на малых осях полные подписи не помещаются,
        # какая кривая что означает, видно из легенды.
        if col == 0:
            ax_pv.set_ylabel("ADC")
            ax_co.set_ylabel("DAC")
        else:
            ax_pv.tick_params(labelleft=False)
            ax_co.tick_params(labelleft=False)
        if row == 2:
            ax_co.set_xlabel("Time (min)")

    fig.align_ylabels()
    handles = ax_pv.get_legend_handles_labels()[0] + ax_co.get_legend_handles_labels()[0]
    fig.legend(handles=handles, loc="outside upper center", ncol=4, frameon=False)

    plt.savefig("review_inference_collage.pdf", bbox_inches="tight", pad_inches=0.02)
    plt.savefig("review_inference_collage.svg", bbox_inches="tight", pad_inches=0.02)
    plt.savefig("review_inference_collage.png", bbox_inches="tight", pad_inches=0.02, dpi=200)
    plt.show()

## Сигнал без стабилизации и с ПИД-регулятором

Те же два рисунка, что в статье для IEEE и в дипломе, но в палитре Okabe-Ito этого блокнота.

- Без стабилизации — `2025-08-28_15-38-42_td3_train_real_async`. Из-за ошибки в прошивке ПИД
  не запускался и напряжение на фазовращатель не подавалось. Линия уставки нарисована для
  сравнения, регулирования в этом прогоне не было.
- С ПИД — `2025-12-02_12-50-33_pid_delta_tuning`. Первые ~41 мин ПИД работал со стартовыми
  коэффициентами (Kp = 7.5, Ki = 10, Kd = 0), дальше их подстраивала нейросеть (TD3).

In [ ]:
from nn_laser_stabilizer.analysis.sources import parse_connection_log, read_tfevents
from nn_laser_stabilizer.utils.paths import get_experiments_dir

FREE_RUN_DIR = get_experiments_dir() / "2025-08-28_15-38-42_td3_train_real_async"
PID_RUN_DIR = get_experiments_dir() / "2025-12-02_12-50-33_pid_delta_tuning"

# Длительности, как в исходных блокнотах; по логам 42.7 мин (15:38:47–16:21:29)
# и 120.6 мин (12:50:33–14:51:12). Время шагов в этих логах не записано,
# поэтому номер шага линейно растягивается на длительность прогона.
FREE_RUN_MIN = 43
PID_RUN_MIN = 121
OLD_SETPOINT = 120  # 1200 / 10 в шкале остальных рисунков

# Без стабилизации: наблюдение в tfevents нормировано на [-1, 1], переводим в отсчёты АЦП.
free_run = read_tfevents(FREE_RUN_DIR / "env_logs").dropna(subset=["Observation/x"])
free_run_pv = (free_run["Observation/x"] + 1) / 2 * 1023
free_run_t_min = free_run["step"] / free_run["step"].max() * FREE_RUN_MIN

# С ПИД: ответы контроллера (READ) из лога соединения; PV в логе умножено на 10.
pid_run = parse_connection_log(PID_RUN_DIR / "connection_logs" / "connection.log")
pid_run = pid_run[pid_run["event"] == "READ"].reset_index(drop=True)
pid_pv = pid_run["process_variable"] / 10
pid_co = pid_run["control_output"]
pid_t_min = pid_run.index / pid_run.index.max() * PID_RUN_MIN

In [ ]:
import pandas as pd

def plot_signals(
    t_min,
    pv: pd.Series,
    out_stem: str,
    *,
    co: pd.Series | None = None,
    setpoint: float = OLD_SETPOINT,
    pv_top: float | None = None,
    t_left: float | None = None,
):
    """Исходный рисунок (сырые сигналы, управление на второй оси Y) в стиле рисунков 2–4.

    pv_top — верхний предел оси фотодетектора, t_left — левый предел оси времени,
    как в исходных рисунках.
    """
    fig, ax_pv = plt.subplots(figsize=(COL_W, 1.7), constrained_layout=True)
    fig.get_layout_engine().set(h_pad=0.02, w_pad=0.02)

    ax_pv.plot(t_min, pv, color=PHOTODETECTOR, alpha=0.7, linewidth=0.5, rasterized=True)
    ax_pv.axhline(y=setpoint, linestyle="--", linewidth=1.2, color=SETPOINT, label="Setpoint")
    ax_pv.set_xlabel("Time (min)")
    ax_pv.set_ylabel("Photodetector signal\n(ADC counts)")
    ax_pv.set_ylim(top=pv_top)
    ax_pv.set_xlim(left=t_left)
    ax_pv.grid(False)

    if co is not None:
        ax_pv.yaxis.label.set_color(PHOTODETECTOR)
        ax_co = ax_pv.twinx()
        ax_co.plot(t_min, co, color=CONTROL, alpha=0.7, linewidth=0.5, rasterized=True)
        ax_co.set_ylabel("Control signal\n(DAC counts)", color=CONTROL)
        ax_co.grid(False)

    ax_pv.legend(
        loc="lower center",
        bbox_to_anchor=(0.5, 1.02),
        frameon=False,
        fontsize=7,
        handlelength=1.6,
        borderpad=0.2,
    )

    plt.savefig(f"{out_stem}.pdf", bbox_inches="tight", pad_inches=0.01)
    plt.savefig(f"{out_stem}.svg", bbox_inches="tight", pad_inches=0.01)
    plt.savefig(f"{out_stem}.png", bbox_inches="tight", pad_inches=0.01, dpi=300)
    plt.show()

In [ ]:
plot_signals(free_run_t_min, free_run_pv, "signal_without_stab")

In [ ]:
plot_signals(pid_t_min, pid_pv, "signals_with_stab", co=pid_co, pv_top=200, t_left=0.1)